In [1]:
import numpy as np
import itertools
from sklearn.linear_model import LassoCV, Lasso
from sklearn.model_selection import KFold

# obtain center of grid
def generate_grid_center(a, b, cell_size, N):
    point = np.arange(a, b, cell_size)
    centers = []
    center = [x + cell_size/2 for x in point]
    for c in center:
        if (a < c < b):
            centers.append(c)
    point_centers = list(itertools.product(centers, repeat=N))
    return point_centers

Ny, Nz = 4, 2
a, b = 0, 32
rho_y, rho_z = 1, 2

# Generate grid and collect centers
centersy = generate_grid_center(a, b, rho_y, Ny)
centersz = generate_grid_center(a, b, rho_z, Nz)

# interconnection map (ground-truth M = average of the components; used
# to label training data and to numerically validate eq. (18) below)
def avg(c):
    return sum(c) / len(c)

print("Now creating dataset")
Data_sety = [(c, avg(c)) for c in centersy]
Data_setz = [(c, avg(c)) for c in centersz]

Xy, z = zip(*Data_sety)
Xz, w = zip(*Data_setz)

lambdas = np.logspace(-3, 3, 9)
k = 5
kf = KFold(n_splits=k, shuffle=True, random_state=42)

print("Now CVing for the best lambda")
lasso_cvy = LassoCV(alphas=lambdas, cv=kf)
lasso_cvy.fit(Xy, z)
lasso_cvz = LassoCV(alphas=lambdas, cv=kf)
lasso_cvz.fit(Xz, w)

best_lambday = lasso_cvy.alpha_
best_lambdaz = lasso_cvz.alpha_
print(f"Best lambda value: y; {best_lambday} and z; {best_lambdaz}")

print("Now fitting model with best lambda")
lassoy = Lasso(alpha=best_lambday)
lassoy.fit(Xy, z)
lassoz = Lasso(alpha=best_lambdaz)
lassoz.fit(Xz, w)

My = lassoy.coef_
print("My:", My)
Mz = lassoz.coef_
print("Mz:", Mz)


# ----------------------------------------------------------------------
# Eq. (18):  \hat{\varepsilon}(\rho_x, w_l, x') := L_M ||\rho_x|| + ||w_l - \bar{M} x'||
# Convention: ||.|| is the sup-norm (Notation section: "||c|| means the
# infinity norm of c").
# ----------------------------------------------------------------------
def eps_hat(rho_x, w_l, x_prime, M_bar, L_M, ord=np.inf):
    x_prime = np.atleast_1d(x_prime)
    rho_x = np.atleast_1d(rho_x)
    fit_residual = np.atleast_1d(w_l - np.dot(M_bar, x_prime))
    return L_M * np.linalg.norm(rho_x, ord=ord) + np.linalg.norm(fit_residual, ord=ord)


def verify_eps_bound(data_set, M_bar, rho_x, L_M, true_map, N,
                      n_random=200, ord=np.inf, seed=0, label="",
                      max_cells=500):
    r"""
    For (up to `max_cells` randomly-subsampled) grid cells (x_l, w_l) in
    `data_set`, checks that
        ||M(x') - \bar{M} x'|| <= \hat{\varepsilon}(\rho_x, w_l, x')
    holds for the 2^N worst-case box corners and n_random random points
    inside Phi_{rho_x}(x_l). Returns the worst-case slack (rhs - lhs);
    negative means a violation.
    """
    rng = np.random.default_rng(seed)
    rho_vec = np.full(N, rho_x, dtype=float)
    worst_slack = np.inf
    n_checked = 0
    n_violations = 0
    eps_min = np.inf
    eps_max = -np.inf

    corners = list(itertools.product([-1.0, 1.0], repeat=N))

    cells = data_set
    if len(data_set) > max_cells:
        idx = rng.choice(len(data_set), size=max_cells, replace=False)
        cells = [data_set[i] for i in idx]

    for x_l, w_l in cells:
        x_l = np.asarray(x_l, dtype=float)

        test_points = [x_l + np.array(c) * rho_vec for c in corners]
        test_points += [
            x_l + rng.uniform(-1.0, 1.0, size=N) * rho_vec
            for _ in range(n_random)
        ]

        for x_prime in test_points:
            lhs = np.abs(true_map(x_prime) - np.dot(M_bar, x_prime))
            rhs = eps_hat(rho_vec, w_l, x_prime, M_bar, L_M, ord=ord)
            slack = rhs - lhs
            n_checked += 1
            if slack < 0:
                n_violations += 1
            worst_slack = min(worst_slack, slack)
            eps_min = min(eps_min, rhs)
            eps_max = max(eps_max, rhs)

    print(f"[{label}] checked {n_checked} points across {len(cells)} cells "
          f"({n_violations} violations, worst-case slack = {worst_slack:.6g})")
    print(f"[{label}] eps_hat min = {eps_min:.6g}, max = {eps_max:.6g}")
    return worst_slack, n_violations


# Try both candidate Lipschitz constants:
#  - L_M = 1: the true, tight sup-norm Lipschitz constant of an average
#    of any number of terms (analytically derived; always valid).
#  - L_M = 1/8: matches the paper's reported value for the related
#    8-subsystem consensus case (Sec. 5.2), which looks like it reports
#    the max absolute entry of \bar{M} rather than the row-sum operator
#    norm. Checking both tells us directly whether 1/8 is too small.
for L_M, tag in [(1.0, "L_M=1"), (1.0/8.0, "L_M=1/8")]:
    print(f"\n--- Using {tag} ---")
    verify_eps_bound(Data_sety, My, rho_y, L_M, avg, Ny, label=f"y-subsystem ({tag})")
    verify_eps_bound(Data_setz, Mz, rho_z, L_M, avg, Nz, label=f"z-subsystem ({tag})")

Now creating dataset
Now CVing for the best lambda
Best lambda value: y; 0.001 and z; 0.001
Now fitting model with best lambda
My: [0.24998827 0.24998827 0.24998827 0.24998827]
Mz: [0.49998824 0.49998824]

--- Using L_M=1 ---
[y-subsystem (L_M=1)] checked 108000 points across 500 cells (0 violations, worst-case slack = 0.99877)
[y-subsystem (L_M=1)] eps_hat min = 1.00001, max = 2.00131
[z-subsystem (L_M=1)] checked 52224 points across 256 cells (0 violations, worst-case slack = 1.9996)
[z-subsystem (L_M=1)] eps_hat min = 2.00002, max = 4.00068

--- Using L_M=1/8 ---
[y-subsystem (L_M=1/8)] checked 108000 points across 500 cells (0 violations, worst-case slack = 0.12377)
[y-subsystem (L_M=1/8)] eps_hat min = 0.125013, max = 1.12631
[z-subsystem (L_M=1/8)] checked 52224 points across 256 cells (0 violations, worst-case slack = 0.249604)
[z-subsystem (L_M=1/8)] eps_hat min = 0.250024, max = 2.25068
